# Middleware: Human In The Loop
<img src="./assets/LC_HITL.png" width="300">



## Setup

In [3]:
from dotenv import load_dotenv
from env_utils import doublecheck_env
from IPython.display import display, Markdown
# Load environment variables from .env
load_dotenv()

# Check and print results
doublecheck_env("example.env")

OPENAI_API_KEY=<not set>
LANGSMITH_API_KEY=****954e
LANGSMITH_TRACING=true
LANGSMITH_PROJECT=****ials


In [4]:
from langchain_community.utilities import SQLDatabase

db = SQLDatabase.from_uri("sqlite:///Chinook.db")

In [5]:
from dataclasses import dataclass

@dataclass
class RuntimeContext:
    db: SQLDatabase

In [6]:
from langchain_core.tools import tool
from langgraph.runtime import get_runtime


@tool
def execute_sql(query: str) -> str:
    """Execute a SQLite command and return results."""
    runtime = get_runtime(RuntimeContext)
    db = runtime.context.db
    
    try:
        return db.run(query)
    except Exception as e:
        return f"Error: {e}"

In [7]:
SYSTEM_PROMPT = """You are a careful SQLite analyst.

Rules:
- Think step-by-step.
- When you need data, call the tool `execute_sql` with ONE SELECT query.
- Read-only only; no INSERT/UPDATE/DELETE/ALTER/DROP/CREATE/REPLACE/TRUNCATE.
- Limit to 5 rows unless the user explicitly asks otherwise.
- If the tool returns 'Error:', revise the SQL and try again.
- Prefer explicit column lists; avoid SELECT *.
- If the database is offline, ask user to try again later without further comment.
"""

## Middleware

In [8]:
import os
from langchain_aws import ChatBedrock
from langchain.agents import create_agent

llm = ChatBedrock(
    model_id="amazon.nova-lite-v1:0",
    model_kwargs=dict(temperature=0),
    region_name=os.getenv("AWS_DEFAULT_REGION", "us-east-1")
)

agent = create_agent(
    model=llm,
    tools=[execute_sql],
    system_prompt=SYSTEM_PROMPT,
    context_schema=RuntimeContext,
)

In [11]:
from langgraph.types import Command

question = "What are the names of all the employees?"

config = {"configurable": {"thread_id": "1"}}

result = agent.invoke(
    {"messages": [{"role": "user", "content": question}]},
    config=config,
    context=RuntimeContext(db=db)
)

if "__interrupt__" in result:
    description = result['__interrupt__'][-1].value['action_requests'][-1]['description']
    print(f"\033[1;3;31m{80 * '-'}\033[0m")
    print(
        f"\033[1;3;31m Interrupt:{description}\033[0m"
    )

    result = agent.invoke(
        Command(
            resume={
                "decisions": [{"type": "reject", "message": "the database is offline."}]
            }
        ),
        config=config,  # Same thread ID to resume the paused conversation
        context=RuntimeContext(db=db),
    )
    print(f"\033[1;3;31m{80 * '-'}\033[0m")

display(Markdown(result["messages"][-1].content))

<thinking> It seems there was an error because the table 'employees' does not exist in the database. I will ask the user to try again later when the database is available. </thinking>

It seems the database is currently offline. Please try again later.

In [13]:
config = {"configurable": {"thread_id": "2"}}

result = agent.invoke(
    {"messages": [{"role": "user", "content": question}]},
    config=config,
    context=RuntimeContext(db=db)
)

while "__interrupt__" in result:
    description = result['__interrupt__'][-1].value['action_requests'][-1]['description']
    print(f"\033[1;3;31m{80 * '-'}\033[0m")
    print(
        f"\033[1;3;31m Interrupt:{description}\033[0m"
    )
    
    result = agent.invoke(
        Command(
            resume={"decisions": [{"type": "approve"}]}
        ),
        config=config,  # Same thread ID to resume the paused conversation
        context=RuntimeContext(db=db),
    )

for msg in result["messages"]:
    display(Markdown(msg.pretty_print()))

================================ Human Message =================================

What are the names of all the employees?


<IPython.core.display.Markdown object>

================================== Ai Message ==================================

[{'type': 'text', 'text': '<thinking> I need to retrieve the names of all the employees from the database. Since the database might have a table that stores employee information, I will first check if such a table exists. </thinking>\n'}, {'type': 'tool_use', 'name': 'execute_sql', 'input': {'query': "SELECT name FROM sqlite_master WHERE type='table';"}, 'id': 'tooluse_DzajlL-XRBOgPLlVehMLSA'}]
Tool Calls:
  execute_sql (tooluse_DzajlL-XRBOgPLlVehMLSA)
 Call ID: tooluse_DzajlL-XRBOgPLlVehMLSA
  Args:
    query: SELECT name FROM sqlite_master WHERE type='table';


<IPython.core.display.Markdown object>

================================= Tool Message =================================
Name: execute_sql

[('Album',), ('Artist',), ('Customer',), ('Employee',), ('Genre',), ('Invoice',), ('InvoiceLine',), ('MediaType',), ('Playlist',), ('PlaylistTrack',), ('Track',)]


<IPython.core.display.Markdown object>

... (2 outputs left out)

================================= Tool Message =================================
Name: execute_sql

[('Andrew', 'Adams'), ('Nancy', 'Edwards'), ('Jane', 'Peacock'), ('Margaret', 'Park'), ('Steve', 'Johnson')]


<IPython.core.display.Markdown object>

================================== Ai Message ==================================

<thinking> The query successfully retrieved the first names and last names of five employees from the database. </thinking> 

Here are the names of five employees from the database:

1. Andrew Adams
2. Nancy Edwards
3. Jane Peacock
4. Margaret Park
5. Steve Johnson

If you need more names or details about the employees, please let me know!


<IPython.core.display.Markdown object>